# Chép lời ghi âm tư vấn trên Kaggle GPU (PhoWhisper-large + lượt hai Whisper medium)

Chạy trên Kaggle (GPU T4) thay cho máy nhà, để máy không nóng.

**Đầu vào:** một Kaggle Dataset **PRIVATE** do Minh tự tải lên, chứa các tệp `*.mp4 / *.m4a / *.mp3`.
Sửa `INPUT_DIR` ở ô dưới cho đúng tên dataset.

**Đầu ra:** `/kaggle/working/asr_out/<stem>.raw.json` và `<stem>.pass2.raw.json`, đúng định dạng của
`backend/scripts/record_batch.py`. Tải thư mục `asr_out` về, chép vào `data/raw/record/out/`, rồi
**che PII trên máy nhà**:

```
docker compose exec api python -m scripts.record_batch import-kaggle --dir data/raw/record/out
```

Bản chép thô CÒN tên người thật: không chia sẻ notebook output công khai. Đặt notebook ở chế độ
Private, và xoá output sau khi đã tải về.

In [ ]:
# Cấu hình
INPUT_DIR = "/kaggle/input"   # Kaggle gắn dataset ở /kaggle/input/datasets/<user>/<slug>: tìm đệ quy
SKIP = {"1069625366079912_AUDIO"}  # tệp 1 đã chép trên máy nhà
# record_inbox.py ghi skip.json (các tệp đã chép ở lượt trước) vào dataset ghi âm
import glob as _g, json as _j
for _f in _g.glob(INPUT_DIR + "/**/skip.json", recursive=True):
    SKIP |= set(_j.load(open(_f)))
OUT_DIR = "/kaggle/working/asr_out"
MAIN_MODEL = "vinai/PhoWhisper-large"   # chuyển sang CTranslate2 fp16 ngay trong notebook
PASS2_MODEL = "medium"                   # faster-whisper, chỉ chạy trên đoạn có số / trích dẫn
LANG = "vi"
BEAM = 5

In [ ]:
!pip -q install "faster-whisper>=1.1.1" "ctranslate2>=4.4" "transformers>=4.44" sentencepiece nvidia-cublas-cu12 "nvidia-cudnn-cu12==9.*"
!nvidia-smi -L

In [ ]:
import os, subprocess
CT2_DIR = "/kaggle/working/phowhisper-large-ct2"
if not os.path.exists(os.path.join(CT2_DIR, "model.bin")):
    cmd = ["ct2-transformers-converter", "--model", MAIN_MODEL, "--output_dir", CT2_DIR,
           "--quantization", "float16", "--copy_files", "tokenizer.json", "preprocessor_config.json"]
    r = subprocess.run(cmd)
    if r.returncode != 0:  # một số bản PhoWhisper thiếu tokenizer.json
        subprocess.run(cmd[:-3] + ["--force"], check=True)
print(os.listdir(CT2_DIR))

In [ ]:
# ctranslate2 cần cuBLAS/cuDNN của CUDA 12; ảnh Kaggle hiện không có libcublas.so.12 (lỗi lần 2):
# cài bánh xe pip của NVIDIA rồi nạp sẵn trước khi import faster_whisper
import ctypes, glob as _glob, os, site
for _pat in ("nvidia/cublas/lib/libcublasLt.so.12", "nvidia/cublas/lib/libcublas.so.12",
             "nvidia/cudnn/lib/libcudnn*.so.9"):
    for _sp in site.getsitepackages():
        for _f in sorted(_glob.glob(os.path.join(_sp, _pat))):
            ctypes.CDLL(_f, mode=ctypes.RTLD_GLOBAL)
import json, re, time, glob
from pathlib import Path
import numpy as np
from faster_whisper import WhisperModel, decode_audio

os.makedirs(OUT_DIR, exist_ok=True)
EXTS = (".mp4", ".m4a", ".mp3", ".wav", ".aac", ".ogg")
files = sorted(p for p in glob.glob(INPUT_DIR + "/**/*", recursive=True) if p.lower().endswith(EXTS))
print(len(files), "tệp")

# cùng luật chọn cửa sổ lượt hai như backend/scripts/record_batch.py (_PASS2)
PASS2_RE = re.compile(r"(?<!\w)(?:điều|khoản|điểm|nghị định|thông tư|luật số|số hiệu|bộ luật|triệu|nghìn|ngàn|tỷ|tỉ"
                      r"|trăm|mươi|phần trăm|năm hai nghìn)(?!\w)", re.IGNORECASE)

def stem_of(path):
    return Path(path).stem.replace(" ", "_")

# PyAV mới bỏ tham số metadata_errors mà faster-whisper.decode_audio còn dùng (lỗi lần 1):
# giải mã bằng ffmpeg -> PCM 16 kHz mono. Giải mã trước mọi tệp bằng 4 tiến trình song song
# để 4 lõi CPU làm việc trong lúc GPU chép lời.
import subprocess
from concurrent.futures import ThreadPoolExecutor

def load_audio(path):
    pcm = subprocess.run(["ffmpeg", "-nostdin", "-v", "error", "-i", path, "-vn", "-ac", "1",
                          "-ar", "16000", "-f", "s16le", "-"], capture_output=True, check=True).stdout
    return np.frombuffer(pcm, dtype=np.int16).astype(np.float32) / 32768.0

_pool = ThreadPoolExecutor(max_workers=4)
AUDIO = {p: _pool.submit(load_audio, p) for p in files if stem_of(p) not in SKIP}


In [ ]:
main = WhisperModel(CT2_DIR, device="cuda", compute_type="float16")
for path in files:
    stem = stem_of(path)
    out = Path(OUT_DIR) / f"{stem}.raw.json"
    if out.exists() or stem in SKIP:
        continue                      # chạy lại được: tệp xong rồi thì bỏ qua
    audio = AUDIO[path].result()
    t0 = time.time()
    segs, _ = main.transcribe(audio, language=LANG, vad_filter=True, word_timestamps=True, beam_size=BEAM)
    rows = [{"start": round(s.start, 2), "end": round(s.end, 2), "text": (s.text or "").strip(),
             "words": [[round(w.start, 2), round(w.end, 2), w.word] for w in (s.words or [])]}
            for s in segs]
    dur = len(audio) / 16000
    json.dump({"file": Path(path).name, "duration_s": dur, "model": MAIN_MODEL, "segments": rows,
               "transcribe_s": round(time.time() - t0), "device": "kaggle-t4"},
              open(out, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    print(stem, len(rows), "đoạn", f"{dur/(time.time()-t0):.1f}x realtime")
del main

In [ ]:
second = WhisperModel(PASS2_MODEL, device="cuda", compute_type="float16")
for path in files:
    stem = stem_of(path)
    raw_p = Path(OUT_DIR) / f"{stem}.raw.json"
    out = Path(OUT_DIR) / f"{stem}.pass2.raw.json"
    if out.exists() or not raw_p.exists():
        continue
    raw = json.load(open(raw_p, encoding="utf-8"))
    windows = []
    for s in raw["segments"]:
        if not PASS2_RE.search(s["text"]):
            continue
        a, b = max(0.0, s["start"] - 1.0), s["end"] + 1.0
        if windows and a - windows[-1][1] < 3.0:
            windows[-1][1] = b
        else:
            windows.append([a, b])
    audio = AUDIO[path].result() if path in AUDIO else load_audio(path)
    rows = []
    for a, b in windows:
        segs, _ = second.transcribe(audio[int(a*16000):int(b*16000)], language=LANG,
                                    vad_filter=False, beam_size=BEAM)
        rows.append({"start": round(a, 2), "end": round(b, 2),
                     "text": " ".join((x.text or "").strip() for x in segs).strip()})
    json.dump({"model": PASS2_MODEL, "windows": rows}, open(out, "w", encoding="utf-8"),
              ensure_ascii=False, indent=1)
    print(stem, len(rows), "cửa sổ lượt hai")

In [ ]:
# Nén để tải về một lần (Output -> asr_out.zip)
!cd /kaggle/working && zip -qr asr_out.zip asr_out && ls -la asr_out asr_out.zip